In [1]:
import torch
import torchinfo
import torchvision

import transformers

/Users/jmanuelc87/Documents/Projects/vision-transformer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights, progress=False)

/Users/jmanuelc87/Documents/Projects/vision-transformer/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
torchinfo.summary(model, input_size=(1,3,224,224), depth=4, row_settings=["var_names"])

Layer (type (var_name))                                      Output Shape              Param #
VisionTransformer (VisionTransformer)                        [1, 1000]                 768
├─Conv2d (conv_proj)                                         [1, 768, 14, 14]          590,592
├─Encoder (encoder)                                          [1, 197, 768]             151,296
│    └─Dropout (dropout)                                     [1, 197, 768]             --
│    └─Sequential (layers)                                   [1, 197, 768]             --
│    │    └─EncoderBlock (encoder_layer_0)                   [1, 197, 768]             --
│    │    │    └─LayerNorm (ln_1)                            [1, 197, 768]             1,536
│    │    │    └─MultiheadAttention (self_attention)         [1, 197, 768]             2,362,368
│    │    │    └─Dropout (dropout)                           [1, 197, 768]             --
│    │    │    └─LayerNorm (ln_2)                            [1, 197, 768]

In [4]:
model

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [5]:
model = transformers.AutoModel.from_pretrained("google/vit-base-patch16-224")

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2173.60it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
torchinfo.summary(model, input_size=(1,3,224,224), depth=4, row_settings=["var_names"])

Layer (type (var_name))                                           Output Shape              Param #
ViTModel (ViTModel)                                               [1, 768]                  --
├─ViTEmbeddings (embeddings)                                      [1, 197, 768]             152,064
│    └─ViTPatchEmbeddings (patch_embeddings)                      [1, 196, 768]             --
│    │    └─Conv2d (projection)                                   [1, 768, 14, 14]          590,592
│    └─Dropout (dropout)                                          [1, 197, 768]             --
├─ViTEncoder (encoder)                                            [1, 197, 768]             --
│    └─ModuleList (layer)                                         --                        --
│    │    └─ViTLayer (0)                                          [1, 197, 768]             --
│    │    │    └─LayerNorm (layernorm_before)                     [1, 197, 768]             1,536
│    │    │    └─ViTAttention (a

In [7]:
model

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d